# 012 — MDOF case-study models and analysis files

Builds the 3-storey EC8-gen2 DC2 concentrically-braced-frame (CBF) case-study models, writes
the OpenSees analysis input files for each one, assembles the flattened design dataset, and
generates the batch launchers that run the analyses.

This is the **first** notebook of the SDOF-parameterisation chain:

| Notebook | Does |
|---|---|
| **012** (this one) | Build MDOF models + analysis files, design dataset, launchers |
| `013-sdof_backbone_parameterisation` | Fit the parameterised SDOF to the MDOF response |
| `014-sdof_validation_and_fragilities` | Validate the SDOF by IDA, produce fragility curves |

**Upstream:** `010_cbfs_reference_designs_ec8_gen2_and_analyses` — supplies the designs in
`cfg["models"]["casestudy_designs_ec8_gen2"]`.

**Downstream:** `013` reads the pushover / cyclic-pushover / modal results produced by the
launchers written here, plus the design dataset written in §3.

> **Note on scope.** This is the EC8-gen2 case-study pipeline, writing to
> `cfg["analysis_data"]["dc2_sdof_fitting"]` (`E:/03_wp1pt4pt1_dc2_sdof_fitting`).
> It is *separate* from `011-create_site_casestudies_and_analyses`, which builds the 60
> site-specific models under `D:/04_site_influence_investigation`. The two pipelines share
> only the batch-launcher output folder, so launcher names must not collide — everything
> here is prefixed `mdof_`, everything in `011` is prefixed `site_`.

## Run order

Run every cell in this notebook top to bottom, then execute the generated launchers from
`cfg["scripts"]["wp1pt4pt1_batch_run"]` in this order:

| # | Launcher | Analysis | Required by |
|---|---|---|---|
| 1 | `mdof_modal.py` | Modal — gives `T1` per building | `013` §1 (equivalent-SDOF conversion) |
| 2 | `mdof_po.py` | Monotonic pushover | `013` §2 |
| 3 | `mdof_cpo.py` | Cyclic pushover (FEMA 461) | `013` §2–§4 (the main backbone fit) |
| 4 | `mdof_ss_modal.py` | Modal, soft-storey variant | `013` §1 (mechanism SDOF) |
| 5 | `mdof_ss_po.py` | Pushover, soft-storey variant | `013` §2 |
| 6 | `mdof_ss_cpo.py` | Cyclic pushover, soft-storey variant | `013` §2 |

Each launcher opens one console per job. Total concurrency is capped globally by
`phd_project/process_semaphore/process_semaphore.py` (physical cores − 3), so the launchers
can safely be started back to back. See `phd_project/scripts/templates/Readme.md` for the
run-script / config contract these generated files rely on.

In [ ]:
%load_ext autoreload
%autoreload 2

## §0 Setup and parameters

In [ ]:
import os
import json
import pickle
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

from phd_project.scripts.templates.copy_templates_to_folders import (
    copy_analysis_config,
    copy_structural_model,
    configure_batch_run_file,
    copy_file,
)
from phd_project.scripts.case_study_design_scripts.design_file_helpers import (
    get_control_node_from_design_file,
    get_n_primary_modes_from_design_file,
    get_n_damping_modes_from_design_file,
    remove_soft_storey_braces,
)
from phd_project.scripts.loading_protocols import get_FEMA461_displacements_for_building

from phd_project.config import config

cfg = config.load_config()

In [ ]:
# ---- Paths -----------------------------------------------------------------
design_file_root = cfg["models"]["casestudy_designs_ec8_gen2"]   # designs from 010
analysis_root_folder = cfg["analysis_data"]["dc2_sdof_fitting"]  # E:/03_wp1pt4pt1_dc2_sdof_fitting
BATCH_ROOT = Path(cfg["scripts"]["wp1pt4pt1_batch_run"])         # where launchers are written

# ---- Flags -----------------------------------------------------------------
# DRY_RUN: report what would be written without touching the filesystem.
DRY_RUN = False
# LIMIT_BUILDINGS: set to a list of tags (e.g. ["3s_cbf_dc2_41"]) for a quick smoke test.
LIMIT_BUILDINGS = None

# ---- Pushover parameters ---------------------------------------------------
max_drift = 6            # %; roof drift limit for the monotonic pushover
drift_step = 0.005       # %; pushover step size
remove_recorders = False # if True, brace elements are deleted when the gussets fail

# ---- Cyclic pushover parameters (FEMA 461 protocol) ------------------------
U_max = 250              # mm
n_levels = 12            # number of amplitude levels in the protocol

# Displacement increment. Some models need a finer step to converge, so the value
# actually in use varies per building -- see CPO_DU_OVERRIDES below. Regenerating a
# folder with the wrong dU will silently undo that tuning, so §2 refuses to shrink a
# config's dU unless you say so.
disp_step = 1.0          # mm -- default for a building with no override

# Per-building overrides, {building_tag: dU}. Populate this from the values already in
# use (the audit cell below prints them) so a rebuild is reproducible.
CPO_DU_OVERRIDES = {}

# If True, §2 keeps the dU already present in an existing config_cyclic_pushover.py
# instead of overwriting it. Leave True unless you deliberately want to re-tune.
PRESERVE_TUNED_DU = True

# ---- NLTHA parameters (optional section at the end, currently unused) ------
gm_json_src_str = "E:/gm_records_p695"

print(f"designs from : {design_file_root}")
print(f"analyses into: {analysis_root_folder}")
print(f"launchers into: {BATCH_ROOT}")

## §1 Discover the case-study buildings

Scan the design folder for completed designs (a subfolder containing `{tag}_out.json`) and
keep only the ductility-class 2 (`dc2`) structures.

In [ ]:
building_tags = sorted(
    f.name
    for f in design_file_root.iterdir()
    if f.is_dir() and (f / f"{f.name}_out.json").exists()
)

# keep only the DC2 structures -- tags look like "3s_cbf_dc2_41"
building_tags = [t for t in building_tags if t.split("_")[2] == "dc2"]

if LIMIT_BUILDINGS is not None:
    building_tags = [t for t in building_tags if t in LIMIT_BUILDINGS]

print(f"{len(building_tags)} DC2 buildings found")
print(building_tags[:5], "..." if len(building_tags) > 5 else "")

## §2 Build the MDOF analysis folders

One folder per building under `analysis_root_folder`, each containing everything OpenSees
needs to be driven by the generated launchers:

| File | Purpose |
|---|---|
| `{tag}_designfile.json` | The design, copied from `010`'s output |
| `structural_model.py` | Model builder, pointed at the design file |
| `run_pushover.py` + `config_pushover.py` | Monotonic pushover to `max_drift` |
| `run_cyclic_pushover.py` + `config_cyclic_pushover.py` | FEMA 461 cyclic pushover |
| `run_modal.py` + `config_modal.py` | Modal analysis (gives `T1`) |

Each run script pairs with a config of the same name — this contract is documented in
`phd_project/scripts/templates/Readme.md`. The job lists built here are turned into
launchers in §5.

### Cyclic-pushover `dU` — audit before overwriting

Several models on disk use a finer displacement increment than the `disp_step` default,
because the coarse step would not converge for them. Rebuilding a folder rewrites
`config_cyclic_pushover.py`, which would throw that tuning away and invalidate the results.

The cell below reports every existing config whose `dU` differs from what would be written.
With `PRESERVE_TUNED_DU = True` (the default) those values are kept; set it to `False` only
if you intend to re-tune from scratch.

In [ ]:
import re

# Only the indented entry inside the `config = {...}` dict counts -- the template also
# mentions "dU" at column 0 inside its explanatory docstring.
_DU_RE = re.compile(r'^[ 	]+"dU"\s*:\s*([0-9.eE+-]+)', re.MULTILINE)


def existing_cpo_du(config_path):
    """dU currently in a config_cyclic_pushover.py, or None if there isn't one."""
    if not config_path.exists():
        return None
    m = _DU_RE.search(config_path.read_text())
    return float(m.group(1)) if m else None


def resolve_cpo_du(building_tag, config_path):
    """dU to write for this building: explicit override > tuned value on disk > default."""
    if building_tag in CPO_DU_OVERRIDES:
        return CPO_DU_OVERRIDES[building_tag]
    if PRESERVE_TUNED_DU:
        current = existing_cpo_du(config_path)
        if current is not None:
            return current
    return disp_step


# ---- audit ----------------------------------------------------------------
divergent = {}
for tag in building_tags:
    cfg_fp = analysis_root_folder / tag / "config_cyclic_pushover.py"
    current = existing_cpo_du(cfg_fp)
    intended = CPO_DU_OVERRIDES.get(tag, disp_step)
    if current is not None and current != intended:
        divergent[tag] = current

if divergent:
    print(f"{len(divergent)} existing config(s) use a dU different from the default "
          f"({disp_step}):")
    for tag, du in sorted(divergent.items()):
        print(f"   {tag:28s} dU = {du}")
    print("\nPRESERVE_TUNED_DU =", PRESERVE_TUNED_DU,
          "-> these will be KEPT." if PRESERVE_TUNED_DU else "-> these will be OVERWRITTEN.")
    print("\nTo make the rebuild reproducible, paste this into CPO_DU_OVERRIDES:")
    print("CPO_DU_OVERRIDES = {")
    for tag, du in sorted(divergent.items()):
        print(f'    "{tag}": {du},')
    print("}")
else:
    print("no divergent dU values found")

In [ ]:
po_jobs = []
cpo_jobs = []
modal_jobs = []

for building_tag in building_tags:

    building_analysis_folder = analysis_root_folder / building_tag

    if DRY_RUN:
        print(f"[dry-run] would build {building_analysis_folder}")
        continue

    building_analysis_folder.mkdir(parents=True, exist_ok=True)

    # ---- design file -------------------------------------------------------
    building_design_file_src = design_file_root / f"{building_tag}/{building_tag}_out.json"
    building_design_file_dst = building_analysis_folder / f"{building_tag}_designfile.json"
    copy_file(building_design_file_src, building_design_file_dst)

    ctrl_node = get_control_node_from_design_file(building_design_file_dst)

    # ---- structural model --------------------------------------------------
    copy_structural_model(
        cfg["templates"]["structural_model"],
        building_analysis_folder / "structural_model.py",
        design_json=building_design_file_dst.name,
        recorder_updates={"incl_remove_recorders": remove_recorders},
        damping_updates={"n_modes": get_n_damping_modes_from_design_file(building_design_file_dst)},
    )

    # ---- monotonic pushover ------------------------------------------------
    po_analysis_dst = building_analysis_folder / "run_pushover.py"
    po_config_dst = building_analysis_folder / "config_pushover.py"

    copy_file(cfg["templates"]["run_pushover"], po_analysis_dst)
    copy_analysis_config(
        cfg["templates"]["config_pushover"],
        po_config_dst,
        results_folder_name="pushover",
        update_config={
            "displacement_type": "drift",
            "U_max": max_drift,
            "dU": drift_step,
            "ctrl_node": ctrl_node,
        },
    )
    po_jobs.append({"script": po_analysis_dst, "config": [po_config_dst]})

    # ---- cyclic pushover ---------------------------------------------------
    cpo_analysis_dst = building_analysis_folder / "run_cyclic_pushover.py"
    cpo_config_dst = building_analysis_folder / "config_cyclic_pushover.py"

    displacements = np.round(
        get_FEMA461_displacements_for_building(building_design_file_dst, n_levels, U_max), 3
    ).tolist()

    copy_file(cfg["templates"]["run_cyclic_pushover"], cpo_analysis_dst)
    copy_analysis_config(
        cfg["templates"]["config_cyclic_pushover"],
        cpo_config_dst,
        results_folder_name="cyclic_pushover",
        update_config={
            "displacement_type": "displacement",
            "dU": resolve_cpo_du(building_tag, cpo_config_dst),
            "ctrl_node": ctrl_node,
            "displacements": displacements,
        },
    )
    cpo_jobs.append({"script": cpo_analysis_dst, "config": [cpo_config_dst]})

    # ---- modal -------------------------------------------------------------
    modal_analysis_dst = building_analysis_folder / "run_modal.py"
    modal_config_dst = building_analysis_folder / "config_modal.py"

    copy_file(cfg["templates"]["run_modal"], modal_analysis_dst)
    copy_analysis_config(
        cfg["templates"]["config_modal"],
        modal_config_dst,
        results_folder_name="modal",
        update_config={
            "n_modes": get_n_primary_modes_from_design_file(building_design_file_dst)
        },
    )
    modal_jobs.append({"script": modal_analysis_dst, "config": [modal_config_dst]})

print(f"built {len(po_jobs)} MDOF analysis folders")

## §3 Design output dataset

Flatten the nested `{tag}_designfile.json` files into a single tidy dataframe — one row per
building, with the design base shear, seismic mass, behaviour factors, spectrum parameters
and storey forces/shears as columns.

This runs **before** any OpenSees analysis: it only reads the design files written in §2.
`013` and `014` both load the resulting pickle.

Only the `building_tags` discovered in §1 are read, which deliberately excludes the
soft-storey `_ss` folders created in §4 — those have their original design file removed and
are not real designs.

In [ ]:
# mapping parameters from design output to columns for df
design_out_parameter_map = {
    "roof_height": "['structure']['level_coordinates'][-1]",
    "V_wind_GQWSI_LC1": "['nonseismic_design_outputs']['GQWSI_LC1']['uls_wind_base_shear']",
    "V_wind_GQWSI_LC2": "['nonseismic_design_outputs']['GQWSI_LC2']['uls_wind_base_shear']",
    "V_wind_GQWSI_LC3": "['nonseismic_design_outputs']['GQWSI_LC3']['uls_wind_base_shear']",
    "T1_GQWSI_LC1": "['nonseismic_design_outputs']['GQWSI_LC1']['period']",
    "T1_GQWSI_LC2": "['nonseismic_design_outputs']['GQWSI_LC2']['period']",
    "T1_GQWSI_LC3": "['nonseismic_design_outputs']['GQWSI_LC3']['period']",
    "gravity_frame_elastic_baseshear": "['seismic_design_outputs']['gravity_frame_elastic_baseshear']",
    "gravity_design_equivalent_seismic_baseshear": "['seismic_design_outputs']['gravity_design_equivalent_seismic_baseshear']",
    "seismic_mass":"['seismic_design_outputs']['seismic_mass']",
    "ductility_class":"['seismic_design_outputs']['ductility_class']",    
    "vertical_regularity":"['seismic_design_outputs']['vertical_regularity']",
    "q_design":"['seismic_design_outputs']['q_design']",
    "q_D":"['seismic_design_outputs']['q_D']",
    "q_R":"['seismic_design_outputs']['q_R']",
    "q_S":"['seismic_design_outputs']['q_S']",
    "q_max":"['seismic_design_outputs']['q_max']",
    "design_period":"['seismic_design_outputs']['design_period']",
    "design_spectral_acceleration":"['seismic_design_outputs']['design_spectral_acceleration']",
    "design_baseshear":"['seismic_design_outputs']['design_baseshear']",
    "lambda":"['seismic_design_outputs']['lambda']",
    "S_alpha_RP":"['seismic_design_outputs']['spectrum_parameters']['S_alpha_RP']",
    "S_beta_RP":"['seismic_design_outputs']['spectrum_parameters']['S_beta_RP']",
    "S_alpha":"['seismic_design_outputs']['spectrum_parameters']['S_alpha']",
    "S_beta":"['seismic_design_outputs']['spectrum_parameters']['S_beta']",
    "T_beta":"['seismic_design_outputs']['spectrum_parameters']['T_beta']",
    "T_A":"['seismic_design_outputs']['spectrum_parameters']['T_A']",
    "T_B":"['seismic_design_outputs']['spectrum_parameters']['T_B']",
    "T_C":"['seismic_design_outputs']['spectrum_parameters']['T_C']",
    "T_D":"['seismic_design_outputs']['spectrum_parameters']['T_D']",
    "F_alpha":"['seismic_design_outputs']['spectrum_parameters']['F_alpha']",
    "F_beta":"['seismic_design_outputs']['spectrum_parameters']['F_beta']",
    "F_T":"['seismic_design_outputs']['spectrum_parameters']['F_T']",
    "F_A":"['seismic_design_outputs']['spectrum_parameters']['F_A']",
    "site_category":"['seismic_design_outputs']['spectrum_parameters']['site_category']",
    "S_delta":"['seismic_design_outputs']['spectrum_parameters']['S_delta']",
    "delta":"['seismic_design_outputs']['spectrum_parameters']['delta']",
    "seismic_action_class":"['seismic_design_outputs']['spectrum_parameters']['seismic_action_class']"
}

In [ ]:
def read_out_value(data, path):
    """Pull a value out of the nested design dict, returning NaN if the path is absent.

    `path` is a string of chained subscripts, e.g. "['structure']['level_coordinates'][-1]".
    """
    try:
        return eval("data" + path)
    except Exception:
        return np.nan

In [ ]:
MAX_STOREYS = 7  # width of the seismic_F* / seismic_V* columns

building_data_dicts = []
for building in building_tags:
    with open(analysis_root_folder / building / f"{building}_designfile.json", "r") as f:
        data = json.load(f)

    building_data = {"name": building, "n_storeys": int(building[0])}
    for df_tag, dd_path in design_out_parameter_map.items():
        building_data[df_tag] = read_out_value(data, dd_path)

    # design storey forces and the cumulative storey shears
    storey_forces = data["seismic_design_outputs"]["storey_forces"]
    storey_shears = np.cumsum(storey_forces)

    for ii in range(MAX_STOREYS):
        building_data[f"seismic_F{ii + 1}"] = (
            storey_forces[ii] if ii < len(storey_forces) else np.nan
        )
        building_data[f"seismic_V{ii + 1}"] = (
            storey_shears[ii] if ii < len(storey_shears) else np.nan
        )

    building_data_dicts.append(building_data)

building_data_df = pd.DataFrame(building_data_dicts)
building_data_df = building_data_df.sort_values(by=["n_storeys", "S_alpha_RP"])
building_data_df = building_data_df.reset_index(drop=True)

if not DRY_RUN:
    with open(cfg["proc_data"]["dc2_casestudy_dataset"], "wb") as f:
        pickle.dump(building_data_df, f)
    print(f"wrote {cfg['proc_data']['dc2_casestudy_dataset']}  ({len(building_data_df)} rows)")

building_data_df.head()

## §4 Soft-storey (`_ss`) variants

Clones of each MDOF model with the braces removed at storeys 1 and 3, forcing a soft-storey
collapse mechanism.

**Why these exist.** The need for them only becomes apparent once you inspect the pushover
and cyclic pushover results in `013` §2 — the as-designed frames do not consistently develop
the single-storey mechanism that the equivalent-SDOF idealisation assumes, so a deliberate
mechanism variant is needed to parameterise the *mechanism* SDOF. They are built here rather
than in `013` because the clone is a purely deterministic transform of the design (copy
folder → remove braces → rewrite `structural_model.py`) and needs nothing from the analysis
results. Building them now means a **single** OpenSees run barrier for all MDOF work instead
of two.

`013` §1 consumes their results as `participation_factors_mechanism.json` and the
`*_mechanism_sdof_parameters.json` files.

In [ ]:
def clone_and_rename_folder(src_path, dst_path, exclude_list, print_successes:bool=False):
    """
    Copies directories in root_path, appends '_ss' to names,
    and excludes specified files/folders.
    """
    # Convert list to a format shutil.copytree understands
    ignore_func = shutil.ignore_patterns(*exclude_list)
           
    # Only process directories that don't already end in _ss
    if os.path.isdir(src_path):       
        
        # If the destination exists, remove it entirely to "replace" it
        if os.path.exists(dst_path):
            if print_successes:
                print(f"Replacing existing folder: {dst_path}")
            shutil.rmtree(dst_path)

        try:
            shutil.copytree(src_path, dst_path, ignore=ignore_func)
            if print_successes:
                print(f"Successfully copied: {src_path} -> {dst_path}")
        except Exception as e:
            print(f"Error copying {src_path}: {e}")

In [ ]:
# Braces are removed at (storey, bay) = (1, 1) and (3, 1).
SS_BRACES_TO_REMOVE = [(1, 1), (3, 1)]

# Results folders and the NLTHA files are not copied -- the clone must be re-analysed.
exclude = [
    "pushover",
    "cyclic_pushover",
    "modal",
    "config_nltha_120121.py",
    "config_nltha_120621.py",
    "run_nltha_ud.py",
]

ss_po_jobs = []
ss_cpo_jobs = []
ss_modal_jobs = []

for building_tag in building_tags:
    folder = analysis_root_folder / building_tag
    new_folder = folder.parent / f"{folder.name}_ss"

    if DRY_RUN:
        print(f"[dry-run] would clone {folder.name} -> {new_folder.name}")
        continue

    clone_and_rename_folder(folder, new_folder, exclude)

    # rebuild the design file with the soft-storey braces removed
    old_filename = f"{folder.name}_designfile.json"
    new_filename = f"{folder.name}_ss_designfile.json"
    remove_soft_storey_braces(
        new_folder / old_filename, new_folder / new_filename, SS_BRACES_TO_REMOVE
    )
    os.remove(new_folder / old_filename)

    # repoint the structural model at the new design file (edited in place)
    copy_structural_model(
        new_folder / "structural_model.py",
        new_folder / "structural_model.py",
        design_json=new_filename,
    )

    ss_po_jobs.append({
        "script": new_folder / "run_pushover.py",
        "config": [new_folder / "config_pushover.py"],
    })
    ss_cpo_jobs.append({
        "script": new_folder / "run_cyclic_pushover.py",
        "config": [new_folder / "config_cyclic_pushover.py"],
    })
    ss_modal_jobs.append({
        "script": new_folder / "run_modal.py",
        "config": [new_folder / "config_modal.py"],
    })

print(f"built {len(ss_po_jobs)} soft-storey variants")

## §5 Write the batch launchers

One launcher per analysis type, written into `cfg["scripts"]["wp1pt4pt1_batch_run"]`.
`configure_batch_run_file` copies `cfg["templates"]["batch_run"]` and rewrites its
`scripts_and_configs` list with the jobs accumulated above.

Naming convention for this folder is `<system>_<analysis>[_<variant>].py`, where `system` is
one of `mdof` / `sdof` / `approx_sdof` (`site_*` is reserved by `011` — do not reuse it).

Jobs whose folder or `structural_model.py` is missing are warned about and skipped rather
than failing the whole write.

In [ ]:
def _valid(jobs, label):
    """Drop jobs whose script, config or structural model is missing, warning about each."""
    kept = []
    for job in jobs:
        folder = job["script"].parent
        missing = [
            p for p in [job["script"], *job["config"], folder / "structural_model.py"]
            if not p.exists()
        ]
        if missing:
            print(f"  ! {label}: skipping {folder.name} -- missing {[m.name for m in missing]}")
            continue
        kept.append(job)
    return kept


batch_run_src = cfg["templates"]["batch_run"]
BATCH_ROOT.mkdir(parents=True, exist_ok=True)

launchers = [
    ("mdof_modal.py", modal_jobs),
    ("mdof_po.py", po_jobs),
    ("mdof_cpo.py", cpo_jobs),
    ("mdof_ss_modal.py", ss_modal_jobs),
    ("mdof_ss_po.py", ss_po_jobs),
    ("mdof_ss_cpo.py", ss_cpo_jobs),
]

for filename, jobs in launchers:
    jobs = _valid(jobs, filename)
    if not jobs:
        print(f"{filename}: no valid jobs -- not written")
        continue
    if DRY_RUN:
        print(f"[dry-run] {filename}: {len(jobs)} jobs")
        continue
    batch_run_dst = BATCH_ROOT / filename
    configure_batch_run_file(batch_run_src, batch_run_dst, jobs)
    print(f"wrote {batch_run_dst.name}: {len(jobs)} jobs")

---

# ⛔ RUN BARRIER — run the MDOF analyses now

Nothing further in this chain works until the OpenSees analyses have been run. From
`cfg["scripts"]["wp1pt4pt1_batch_run"]`, execute:

```
python mdof_modal.py        # 1. fast -- needed first, 013 §1 uses T1
python mdof_po.py           # 2.
python mdof_cpo.py          # 3. the slow one; this is the main input to 013
python mdof_ss_modal.py     # 4.
python mdof_ss_po.py        # 5.
python mdof_ss_cpo.py       # 6. also slow
```

Each launcher spawns one console per building. Concurrency is capped globally by
`phd_project/process_semaphore/process_semaphore.py`, so starting them back to back is safe
— jobs queue rather than oversubscribe the machine.

Expect results at:

- `<analysis_root_folder>/<tag>/modal/modal_properties.json`
- `<analysis_root_folder>/<tag>/pushover/po_curve.csv`
- `<analysis_root_folder>/<tag>/cyclic_pushover/po_curve.csv`

and the same three under each `<tag>_ss/`.

**Once these have completed, continue in `013-sdof_backbone_parameterisation.ipynb`.**

---

## Appendix — optional NLTHA setup (not currently used)

Kept for reference. This wrote a nonlinear time-history analysis config per record for two
FEMA P695 ground motions. It is not part of the SDOF-parameterisation chain — the IDA in
`014` supersedes it. Uncomment and run after §2 if you need it.

In [ ]:
# nltha_jobs = []
#
# for building_tag in building_tags:
#     building_analysis_folder = analysis_root_folder / building_tag
#
#     nltha_analysis_dst = building_analysis_folder / "run_nltha_ud.py"
#     copy_file(cfg["templates"]["run_nltha_ud"], nltha_analysis_dst)
#
#     for record, sf in [("fema_p695_120121.json", 2.0), ("fema_p695_120621.json", 2.6)]:
#         rec_num = record.split(".json")[0].split("_")[-1]
#         nltha_config_dst = building_analysis_folder / f"config_nltha_{rec_num}.py"
#
#         copy_analysis_config(
#             cfg["templates"]["config_nltha"],
#             nltha_config_dst,
#             results_folder_name=f"nltha_{rec_num}_sf{int(sf * 1000)}",
#             update_config={"gm_json_file": record, "scale_factor": sf},
#             gm_json_src_str=gm_json_src_str,
#         )
#         nltha_jobs.append({"script": nltha_analysis_dst, "config": [nltha_config_dst]})
#
# configure_batch_run_file(batch_run_src, BATCH_ROOT / "mdof_nltha.py", nltha_jobs)